In [2]:
import pandas as pd
import numpy as np

In [3]:
df_tw = pd.read_csv(r'C:\Users\afpue\OneDrive\Documentos\GitHub\icare\archivos\df_twitter.csv')

In [4]:
df_tw.columns

Index(['Author', 'Content', 'Date', 'Location', 'Number of Likes',
       'Number of Retweets', 'In Reply To', 'Author Name',
       'Author Description', 'Author Statuses Count',
       'Author Favourites Count', 'Author Friends Count',
       'Author Followers Count', 'Author Listed Count', 'Author Verified',
       'Mentions', 'Hashtags', 'Content_cleaned', 'Content_cleaned_2',
       'Seed_Set_1', 'In Reply To Normalized', 'Seed_Set_2', 'Seed_Set_2_Prob',
       'Seed_Set_2_Model', 'Author_Normalized', 'Seed_Set_3', 'Entidad',
       'Prob entidad', 'Polaridad', 'Fecha', 'Sentimiento'],
      dtype='str')

In [ ]:
"""
=============================================================================
CIENCIA EN LA ESFERA PÚBLICA - ADAPTACIÓN A TWEETS EN ESPAÑOL
Replicación de la metodología: Gómez-Montoya & Suárez-Sierra (2026)

Flujo:
  Recolección de datos
    → Preprocesamiento (Tokenización, Stopwords, Lematización)
    → Identificación de referencias a SALUD
        (Embeddings, Matriz de similitud, Umbral)
    → Corpus y subcorpus de salud
    → Análisis comparativo
        (Descriptivo/exploratorio, NER, Topic Modeling, Vocabulario/POS)
    → Interpretación y discusión
=============================================================================
"""

# ─────────────────────────────────────────────────────────────────────────────
# 0.  CONFIGURACIÓN GENERAL
# ─────────────────────────────────────────────────────────────────────────────

import os
import re
import json
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")

# Semilla de reproducibilidad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ── Rutas de salida ──────────────────────────────────────────────────────────
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Dispositivo (GPU si está disponible) ────────────────────────────────────
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[CONFIG] Dispositivo: {DEVICE}")


# =============================================================================
# PASO 1 – RECOLECCIÓN DE DATOS
# =============================================================================
print("\n" + "=" * 70)
print("PASO 1 – RECOLECCIÓN DE DATOS")
print("=" * 70)

# Se asume que el DataFrame ya fue cargado previamente como df_tw.
# Ejemplo:
#   import pandas as pd
#   df_tw = pd.read_csv("tweets.csv")

# Columnas relevantes que usaremos a lo largo del análisis
COL_ID      = "Author"           # identificador único por tweet
COL_TEXT    = "Content_cleaned"  # texto preprocesado básico disponible
COL_DATE    = "Fecha"            # columna de fecha ya parseada
COL_AUTHOR  = "Author_Normalized"

# Verificación rápida de la carga
assert "df_tw" in dir(), (
    "ERROR: El DataFrame 'df_tw' no está definido. "
    "Cárgalo antes de correr este script."
)

print(f"  Tweets cargados  : {len(df_tw):,}")
print(f"  Columnas         : {list(df_tw.columns)}")
print(f"  Rango de fechas  : {df_tw[COL_DATE].min()} → {df_tw[COL_DATE].max()}")

# Guardamos una copia de trabajo para no alterar el original
df = df_tw.copy()

# Asignamos un ID único si no existe
df["tweet_id"] = range(len(df))

In [ ]:
# =============================================================================
# PASO 2 – PREPROCESAMIENTO
# =============================================================================
print("\n" + "=" * 70)
print("PASO 2 – PREPROCESAMIENTO")
print("=" * 70)

# ── 2.1  Limpieza inicial ────────────────────────────────────────────────────
print("\n  [2.1] Limpieza de texto (URLs, menciones, hashtags, caracteres especiales)...")

def limpiar_tweet(texto: str) -> str:
    """
    Elimina URLs, menciones (@usuario), hashtags (#tag),
    espacios redundantes y caracteres no alfabéticos/numéricos.
    Equivalente a la etapa de limpieza del paper (sección Preprocessing).
    """
    if not isinstance(texto, str):
        return ""
    # URLs
    texto = re.sub(r"http\S+|www\.\S+", " ", texto)
    # Menciones
    texto = re.sub(r"@\w+", " ", texto)
    # Hashtags (conservamos la palabra sin el símbolo)
    texto = re.sub(r"#(\w+)", r"\1", texto)
    # RT al inicio
    texto = re.sub(r"^RT\s+", "", texto, flags=re.IGNORECASE)
    # Caracteres que no sean letras, números ni puntuación básica
    texto = re.sub(r"[^\w\sáéíóúüñÁÉÍÓÚÜÑ.,;:!?¿¡\-]", " ", texto)
    # Espacios múltiples
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

df["texto_limpio"] = df[COL_TEXT].apply(limpiar_tweet)

n_vacios = (df["texto_limpio"].str.strip() == "").sum()
print(f"    Tweets con texto vacío tras limpieza: {n_vacios}")
df = df[df["texto_limpio"].str.strip() != ""].reset_index(drop=True)
print(f"    Tweets conservados                  : {len(df):,}")

# ── 2.2  Tokenización ────────────────────────────────────────────────────────
print("\n  [2.2] Tokenización...")

import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
from nltk.tokenize import word_tokenize

def tokenizar(texto: str) -> list[str]:
    """Divide el texto en tokens (palabras)."""
    return word_tokenize(texto, language="spanish")

df["tokens"] = df["texto_limpio"].apply(tokenizar)
n_tokens_total = df["tokens"].apply(len).sum()
print(f"    Tokens totales: {n_tokens_total:,}")

# ── 2.3  Eliminación de stopwords ────────────────────────────────────────────
print("\n  [2.3] Eliminación de stopwords...")

from nltk.corpus import stopwords
nltk.download("stopwords", quiet=True)

stop_es = set(stopwords.words("spanish"))
# Ampliamos con jerga de redes sociales
stop_custom = {"rt", "via", "q", "xq", "pq", "tb", "tmb", "d", "x", "jajaja",
               "jaja", "jeje", "xa", "xd", "nd", "nada", "si", "no", "hay",
               "ser", "haber", "estar", "tener", "hacer", "ir", "ver", "dar"}
STOPWORDS = stop_es | stop_custom

def eliminar_stopwords(tokens: list[str]) -> list[str]:
    return [t for t in tokens if t.lower() not in STOPWORDS and len(t) > 1]

df["tokens_sin_stop"] = df["tokens"].apply(eliminar_stopwords)
print(f"    Stopwords configuradas: {len(STOPWORDS)}")

# ── 2.4  Lematización (Stanford Stanza) ─────────────────────────────────────
print("\n  [2.4] Lematización con Stanza (igual que en el paper)...")

import stanza
# Descarga silenciosa del modelo de español si no existe
stanza.download("es", verbose=False)
nlp_stanza = stanza.Pipeline(
    lang="es",
    processors="tokenize,mwt,pos,lemma",
    use_gpu=(DEVICE == "cuda"),
    verbose=False,
)

def lematizar_batch(textos: list[str], batch_size: int = 512) -> list[str]:
    """
    Lematiza una lista de textos usando Stanza en lotes para eficiencia.
    Devuelve un texto por tweet con los lemas concatenados.
    """
    lemas = []
    for i in range(0, len(textos), batch_size):
        lote = textos[i : i + batch_size]
        docs = [stanza.Document([], text=t) for t in lote]
        docs = nlp_stanza(docs)
        for doc in docs:
            lemas_doc = [
                word.lemma.lower()
                for sent in doc.sentences
                for word in sent.words
                if word.lemma and word.lemma.lower() not in STOPWORDS
                and len(word.lemma) > 1
            ]
            lemas.append(" ".join(lemas_doc))
        if (i // batch_size) % 10 == 0:
            print(f"      Lote {i // batch_size + 1} procesado ({i + len(lote):,}/{len(textos):,})")
    return lemas

df["texto_lematizado"] = lematizar_batch(df["texto_limpio"].tolist())
print(f"    Lematización completada para {len(df):,} tweets.")

In [6]:
# =============================================================================
# PASO 3 – IDENTIFICACIÓN DE REFERENCIAS A SALUD
# =============================================================================
print("\n" + "=" * 70)
print("PASO 3 – IDENTIFICACIÓN DE REFERENCIAS A SALUD")
print("=" * 70)

# ── 3.1  Definición del Tesauro UNESCO – Salud ───────────────────────────────
print("\n  [3.1] Definición del Tesauro UNESCO – Salud...")

TESAURO_SALUD: dict[str, list[str]] = {
    "Economía de la salud": [
        "economía de la salud", "financiamiento sanitario", "gasto en salud",
        "seguro médico", "seguridad social", "costos hospitalarios",
        "privatización salud", "presupuesto salud", "acceso a medicamentos",
        "cobertura universal", "subsidio salud", "tarifa médica",
    ],
    "Estadísticas sanitarias": [
        "estadísticas sanitarias", "datos epidemiológicos", "indicadores de salud",
        "mortalidad", "morbilidad", "tasa de incidencia", "prevalencia",
        "esperanza de vida", "registros clínicos", "vigilancia epidemiológica",
        "encuestas de salud", "datos demográficos salud",
    ],
    "Higiene": [
        "higiene", "higiene personal", "saneamiento básico", "lavado de manos",
        "desinfección", "esterilización", "agua potable", "higiene alimentaria",
        "higiene pública", "salubridad", "condiciones sanitarias",
        "prevención enfermedades",
    ],
    "Control de alimentos": [
        "control de alimentos", "inocuidad alimentaria", "seguridad alimentaria",
        "inspección alimentos", "contaminación alimentos", "etiquetado nutricional",
        "regulación alimentaria", "INVIMA", "alimentos ultraprocesados",
        "calidad nutricional", "cadena alimentaria", "aditivos alimentarios",
    ],
    "Epidemiología": [
        "epidemiología", "brote epidémico", "pandemia", "epidemia", "endemia",
        "transmisión de enfermedades", "contagio", "infección", "vectores",
        "cadena de transmisión", "R0", "tasa de contagio", "cuarentena",
        "aislamiento preventivo", "trazabilidad", "contacto estrecho",
    ],
    "Higiene ambiental": [
        "higiene ambiental", "contaminación ambiental", "calidad del aire",
        "residuos sólidos", "gestión de residuos", "contaminación del agua",
        "ruido ambiental", "exposición a químicos", "salud ambiental",
        "ecosistemas y salud", "plaguicidas", "metales pesados",
    ],
    "Lucha contra las enfermedades": [
        "lucha contra enfermedades", "control de enfermedades", "vacunación",
        "inmunización", "erradicación enfermedades", "tratamiento médico",
        "diagnóstico temprano", "detección oportuna", "campaña de salud",
        "programa de salud pública", "resistencia antimicrobiana", "antibióticos",
    ],
    "Política sobre drogas": [
        "política de drogas", "regulación de medicamentos", "medicamentos genéricos",
        "acceso a medicamentos", "precios de medicamentos", "ensayos clínicos",
        "aprobación de fármacos", "farmacovigilancia", "patentes farmacéuticas",
        "automedicación", "uso racional de medicamentos",
    ],
    "Lucha contra la toxicomanía": [
        "toxicomanía", "adicción", "consumo de drogas", "sustancias psicoactivas",
        "dependencia", "rehabilitación", "desintoxicación", "alcoholismo",
        "drogadicción", "prevención consumo drogas", "tratamiento adicciones",
        "reducción de daños",
    ],
    "Salud de la mujer": [
        "salud de la mujer", "salud femenina", "derechos reproductivos",
        "salud sexual reproductiva", "anticoncepción", "planificación familiar",
        "cáncer de mama", "cáncer de cuello uterino", "violencia obstétrica",
        "lactancia materna", "menstruación", "menopausia",
    ],
    "Salud materno-infantil": [
        "salud materno-infantil", "salud materna", "mortalidad materna",
        "atención prenatal", "parto", "neonatal", "lactancia", "mortalidad infantil",
        "vacunas infantiles", "nutrición infantil", "desarrollo infantil",
        "pediatría", "crecimiento y desarrollo",
    ],
    "Salud mental": [
        "salud mental", "bienestar mental", "trastorno mental", "depresión",
        "ansiedad", "estrés", "burnout", "suicidio", "prevención suicidio",
        "psicología clínica", "psiquiatría", "terapia psicológica",
        "enfermedad mental", "psicosis", "esquizofrenia", "trastorno bipolar",
    ],
}

N_CATEGORIAS = len(TESAURO_SALUD)
print(f"    Categorías del Tesauro: {N_CATEGORIAS}")
for cat, terminos in TESAURO_SALUD.items():
    print(f"      · {cat}: {len(terminos)} términos")

# ── 3.2  Selección y carga del modelo de embeddings ─────────────────────────
print("\n  [3.2] Carga del modelo de embeddings (multilingüe, optimizado para similitud)...")
# Usamos paraphrase-multilingual-mpnet-base-v2 (≈278M parámetros),
# equivalente funcional al embeddinggemma-300m del paper,
# con fuerte desempeño en similitud semántica en español.

from sentence_transformers import SentenceTransformer

MODELO_EMBEDDINGS = "paraphrase-multilingual-mpnet-base-v2"
modelo_emb = SentenceTransformer(MODELO_EMBEDDINGS, device=DEVICE)
print(f"    Modelo cargado: {MODELO_EMBEDDINGS} | Dispositivo: {DEVICE}")

# ── 3.3  Segmentación en fragmentos (chunks) ─────────────────────────────────
print("\n  [3.3] Segmentación en fragmentos...")
# Los tweets son cortos (≤280 caracteres ≈ 50-60 palabras).
# Usamos el tweet completo como unidad mínima (no se subdivide).
# En caso de threads (tweets muy largos), aplicamos un chunk de 60 palabras
# con solapamiento de 10 palabras, siguiendo el espíritu del paper.

CHUNK_WORDS  = 60
OVERLAP      = 10
MIN_WORDS    = 10

def chunkear_texto(tweet_id: int, texto: str) -> list[dict]:
    """
    Divide el texto de un tweet en fragmentos.
    Para tweets cortos, devuelve el tweet íntegro como un solo fragmento.
    """
    palabras = texto.split()
    if len(palabras) <= CHUNK_WORDS:
        return [{"tweet_id": tweet_id, "chunk_id": 0, "texto": texto}]
    chunks = []
    paso = CHUNK_WORDS - OVERLAP
    for i, inicio in enumerate(range(0, len(palabras), paso)):
        segmento = palabras[inicio : inicio + CHUNK_WORDS]
        if len(segmento) < MIN_WORDS:
            # Fusionar con el chunk anterior
            if chunks:
                chunks[-1]["texto"] += " " + " ".join(segmento)
            break
        chunks.append({
            "tweet_id": tweet_id,
            "chunk_id": i,
            "texto": " ".join(segmento),
        })
    return chunks

registros_chunks = []
for _, fila in df.iterrows():
    registros_chunks.extend(
        chunkear_texto(fila["tweet_id"], fila["texto_lematizado"])
    )

df_chunks = pd.DataFrame(registros_chunks)
print(f"    Fragmentos generados: {len(df_chunks):,} (de {len(df):,} tweets)")

# ── 3.4  Generación de embeddings ────────────────────────────────────────────
print("\n  [3.4] Generación de embeddings para fragmentos y categorías del Tesauro...")

# Embeddings de los fragmentos
BATCH_SIZE_EMB = 256
print(f"    Codificando {len(df_chunks):,} fragmentos (batch={BATCH_SIZE_EMB})...")

embeddings_chunks = modelo_emb.encode(
    df_chunks["texto"].tolist(),
    batch_size=BATCH_SIZE_EMB,
    show_progress_bar=True,
    device=DEVICE,
    normalize_embeddings=True,  # facilita el cómputo de coseno
)
print(f"    Shape embeddings fragmentos: {embeddings_chunks.shape}")

# Embeddings de las categorías del Tesauro
nombres_cats  = list(TESAURO_SALUD.keys())
textos_cats   = [
    nombre + " " + " ".join(terminos)
    for nombre, terminos in TESAURO_SALUD.items()
]

embeddings_cats = modelo_emb.encode(
    textos_cats,
    batch_size=N_CATEGORIAS,
    show_progress_bar=False,
    device=DEVICE,
    normalize_embeddings=True,
)
print(f"    Shape embeddings categorías: {embeddings_cats.shape}")

# ── 3.5  Matriz de similitud coseno ─────────────────────────────────────────
print("\n  [3.5] Cómputo de la matriz de similitud coseno S ∈ R^(m × n)...")
# Como los embeddings están L2-normalizados, similitud coseno = producto punto.

from sklearn.metrics.pairwise import cosine_similarity

S = cosine_similarity(embeddings_chunks, embeddings_cats)   # (m, n)
print(f"    Matriz S: {S.shape}  →  {S.shape[0]:,} fragmentos × {S.shape[1]} categorías")
print(f"    Similitud máxima global : {S.max():.4f}")
print(f"    Similitud media global  : {S.mean():.4f}")

# Asignación de categoría con máxima similitud por fragmento
idx_max  = S.argmax(axis=1)                     # j*(i)
val_max  = S.max(axis=1)                        # max_j s_{i,j}

df_chunks["cat_idx"]      = idx_max
df_chunks["cat_nombre"]   = [nombres_cats[j] for j in idx_max]
df_chunks["sim_max"]      = val_max

# ── 3.6  Selección del umbral τ ───────────────────────────────────────────────
print("\n  [3.6] Determinación del umbral τ mediante validación manual...")

# Distribución de similitudes para orientar el umbral inicial
q_vals = [0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
print("    Distribución de sim_max:")
for q in q_vals:
    print(f"      P{int(q*100):3d}: {np.quantile(val_max, q):.4f}")

UMBRAL_INICIAL = 0.40
df_filtrado = df_chunks[df_chunks["sim_max"] >= UMBRAL_INICIAL].copy()

muestra_por_cat = 10

# ── FIX: muestreo robusto sin depender del MultiIndex que genera groupby ──
frames = []
for cat, grupo in df_filtrado.groupby("cat_nombre"):
    frames.append(grupo.sample(min(len(grupo), muestra_por_cat), random_state=SEED))

df_muestra_val = pd.concat(frames, ignore_index=True)[
    ["tweet_id", "chunk_id", "cat_nombre", "sim_max", "texto"]
]

ruta_muestra = OUTPUT_DIR / "df_muestra_validacion.csv"
df_muestra_val.to_csv(ruta_muestra, index=False, encoding="utf-8-sig")
print(f"\n    Muestra guardada en: {ruta_muestra}")
print(f"    ({len(df_muestra_val)} fragmentos | ~{muestra_por_cat} por categoría)")
print(
    "\n    *** ACCIÓN REQUERIDA ***\n"
    "    Abre el archivo 'df_muestra_validacion.csv', añade la columna\n"
    "    'es_salud' (1 = mención real, 0 = no mención) y guárdalo como\n"
    "    'df_muestra_validacion_anotada.csv'.\n"
    "    Luego ejecuta el bloque de cálculo de exactitud.\n"
)

# >>> CÁLCULO DE EXACTITUD (ejecutar tras anotar el CSV) ─────────────────────
ruta_anotada = OUTPUT_DIR / "df_muestra_validacion_anotada.csv"

if ruta_anotada.exists():
    df_anotada = pd.read_csv(ruta_anotada)
    assert "es_salud" in df_anotada.columns, "Falta la columna 'es_salud'."

    taus = np.arange(0.40, 0.85, 0.01)
    exactitudes = []
    for tau in taus:
        pred = (df_anotada["sim_max"] >= tau).astype(int)
        acc  = (pred == df_anotada["es_salud"]).mean()
        exactitudes.append(acc)

    tau_optimo = taus[np.argmax(exactitudes)]
    acc_optima = max(exactitudes)
    print(f"\n    τ óptimo : {tau_optimo:.2f}  |  Exactitud: {acc_optima:.3f}")

    # Gráfica de exactitud vs. umbral
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(taus, exactitudes, color="#0077b6", linewidth=2)
    ax.axvline(tau_optimo, color="#e63946", linestyle="--",
               label=f"Mejor umbral {tau_optimo:.2f}")
    ax.set_xlabel("Umbral (τ)", fontsize=12)
    ax.set_ylabel("Exactitud", fontsize=12)
    ax.set_title("Exactitud como función del umbral τ", fontsize=13)
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "figura_umbral.png", dpi=150)
    plt.close()
    print(f"    Figura guardada: {OUTPUT_DIR / 'figura_umbral.png'}")

else:
    # Umbral por defecto mientras se anota la muestra
    tau_optimo = 0.47
    print(f"\n    Archivo anotado no encontrado. Se usará τ = {tau_optimo} (valor por defecto).")

TAU = tau_optimo

# ── 3.7  Construcción del subcorpus de salud ─────────────────────────────────
print(f"\n  [3.7] Aplicando umbral τ = {TAU:.2f} al corpus completo...")

df_chunks["es_salud"] = (df_chunks["sim_max"] >= TAU).astype(int)
df_subcorpus = df_chunks[df_chunks["es_salud"] == 1].copy()

print(f"    Fragmentos con mención de salud : {len(df_subcorpus):,}")
print(f"    Porcentaje del total            : {len(df_subcorpus)/len(df_chunks)*100:.1f}%")

# Tweet-level: tweet contiene al menos un fragmento de salud
ids_salud  = set(df_subcorpus["tweet_id"].unique())
df["tiene_salud"] = df["tweet_id"].apply(lambda x: 1 if x in ids_salud else 0)
n_tweets_salud = df["tiene_salud"].sum()
pct_tweets = n_tweets_salud / len(df) * 100
print(f"\n    Tweets con ≥1 mención de salud : {n_tweets_salud:,} ({pct_tweets:.1f}%)")

# Distribución por categoría del Tesauro
print("\n    Distribución por categoría:")
dist_cats = (
    df_subcorpus["cat_nombre"]
    .value_counts()
    .reset_index()
    .rename(columns={"cat_nombre": "Categoría", "count": "N_fragmentos"})
)
dist_cats["Porcentaje"] = (dist_cats["N_fragmentos"] / len(df_subcorpus) * 100).round(2)
print(dist_cats.to_string(index=False))
dist_cats.to_csv(OUTPUT_DIR / "distribucion_categorias_salud.csv", index=False)




PASO 3 – IDENTIFICACIÓN DE REFERENCIAS A SALUD

  [3.1] Definición del Tesauro UNESCO – Salud...
    Categorías del Tesauro: 12
      · Economía de la salud: 12 términos
      · Estadísticas sanitarias: 12 términos
      · Higiene: 12 términos
      · Control de alimentos: 12 términos
      · Epidemiología: 16 términos
      · Higiene ambiental: 12 términos
      · Lucha contra las enfermedades: 12 términos
      · Política sobre drogas: 11 términos
      · Lucha contra la toxicomanía: 12 términos
      · Salud de la mujer: 12 términos
      · Salud materno-infantil: 13 términos
      · Salud mental: 16 términos

  [3.2] Carga del modelo de embeddings (multilingüe, optimizado para similitud)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    Modelo cargado: paraphrase-multilingual-mpnet-base-v2 | Dispositivo: cuda

  [3.3] Segmentación en fragmentos...
    Fragmentos generados: 151,424 (de 151,424 tweets)

  [3.4] Generación de embeddings para fragmentos y categorías del Tesauro...
    Codificando 151,424 fragmentos (batch=256)...


Batches:   0%|          | 0/592 [00:00<?, ?it/s]

    Shape embeddings fragmentos: (151424, 768)
    Shape embeddings categorías: (12, 768)

  [3.5] Cómputo de la matriz de similitud coseno S ∈ R^(m × n)...
    Matriz S: (151424, 12)  →  151,424 fragmentos × 12 categorías
    Similitud máxima global : 0.8283
    Similitud media global  : 0.1868

  [3.6] Determinación del umbral τ mediante validación manual...
    Distribución de sim_max:
      P 10: 0.1930
      P 25: 0.2417
      P 50: 0.3193
      P 75: 0.4197
      P 90: 0.4975
      P 95: 0.5405
      P 99: 0.6180

    Muestra guardada en: outputs\df_muestra_validacion.csv
    (120 fragmentos | ~10 por categoría)

    *** ACCIÓN REQUERIDA ***
    Abre el archivo 'df_muestra_validacion.csv', añade la columna
    'es_salud' (1 = mención real, 0 = no mención) y guárdalo como
    'df_muestra_validacion_anotada.csv'.
    Luego ejecuta el bloque de cálculo de exactitud.


    Archivo anotado no encontrado. Se usará τ = 0.47 (valor por defecto).

  [3.7] Aplicando umbral τ = 0.47 al corp

In [7]:
# =============================================================================
# PASO 4 – ANÁLISIS COMPARATIVO
# =============================================================================
print("\n" + "=" * 70)
print("PASO 4 – ANÁLISIS COMPARATIVO")
print("=" * 70)

# ─────────────────────────────────────────────────────────────────────────────
# 4a. ANÁLISIS DESCRIPTIVO Y EXPLORATORIO
# ─────────────────────────────────────────────────────────────────────────────
print("\n  [4a] Análisis descriptivo y exploratorio...")

# Estadísticas generales
df["n_palabras"] = df["texto_limpio"].apply(lambda x: len(x.split()))

stats = {
    "Total tweets"                        : len(df),
    "Total palabras"                      : df["n_palabras"].sum(),
    "Media palabras por tweet"            : round(df["n_palabras"].mean(), 1),
    "Mediana palabras por tweet"          : df["n_palabras"].median(),
    "Máximo palabras"                     : df["n_palabras"].max(),
    "Mínimo palabras"                     : df["n_palabras"].min(),
    "Autores distintos"                   : df[COL_AUTHOR].nunique(),
    "Tweets con mención de salud"         : int(df["tiene_salud"].sum()),
    "% tweets con mención de salud"       : round(pct_tweets, 1),
}

print("\n    ESTADÍSTICAS GENERALES DEL CORPUS")
print("    " + "-" * 45)
for k, v in stats.items():
    print(f"    {k:<40}: {v}")

pd.DataFrame(stats.items(), columns=["Estadístico", "Valor"]).to_csv(
    OUTPUT_DIR / "estadisticas_generales.csv", index=False
)

# Nubes de palabras
from wordcloud import WordCloud

def generar_nube(textos: list[str], titulo: str, ruta: Path, max_words: int = 200):
    texto_completo = " ".join(textos)
    wc = WordCloud(
        width=900, height=500, background_color="white",
        colormap="Blues", max_words=max_words,
        stopwords=STOPWORDS, collocations=False,
    ).generate(texto_completo)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.imshow(wc, interpolation="bilinear")
    ax.axis("off")
    ax.set_title(titulo, fontsize=14, pad=12)
    fig.tight_layout()
    fig.savefig(ruta, dpi=150)
    plt.close()
    print(f"    Nube guardada: {ruta}")

textos_corpus   = df["texto_lematizado"].tolist()
textos_subcorpus = df.loc[df["tiene_salud"] == 1, "texto_lematizado"].tolist()

generar_nube(textos_corpus,
             "Nube de palabras – Corpus completo",
             OUTPUT_DIR / "nube_corpus_completo.png")

generar_nube(textos_subcorpus,
             "Nube de palabras – Subcorpus de salud",
             OUTPUT_DIR / "nube_subcorpus_salud.png")

# Distribución temporal (mensual) de menciones de salud
if COL_DATE in df.columns:
    df[COL_DATE] = pd.to_datetime(df[COL_DATE], errors="coerce")
    df["anio_mes"] = df[COL_DATE].dt.to_period("M")

    temporal = (
        df.groupby("anio_mes")
        .agg(total=("tweet_id", "count"), salud=("tiene_salud", "sum"))
        .reset_index()
    )
    temporal["pct_salud"] = (temporal["salud"] / temporal["total"] * 100).round(1)
    temporal["anio_mes_str"] = temporal["anio_mes"].astype(str)

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(temporal["anio_mes_str"], temporal["pct_salud"],
            marker="o", linewidth=2, color="#0077b6", markersize=5)
    ax.fill_between(temporal["anio_mes_str"], temporal["pct_salud"],
                    alpha=0.15, color="#0077b6")
    ax.set_xlabel("Mes", fontsize=11)
    ax.set_ylabel("% tweets con mención de salud", fontsize=11)
    ax.set_title("Distribución mensual de tweets con referencias a salud", fontsize=13)
    ax.tick_params(axis="x", rotation=45)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "figura_temporal.png", dpi=150)
    plt.close()
    print("    Figura temporal guardada.")
    temporal.to_csv(OUTPUT_DIR / "distribucion_temporal.csv", index=False)

# ─────────────────────────────────────────────────────────────────────────────
# 4b. RECONOCIMIENTO DE ENTIDADES (NER)
# ─────────────────────────────────────────────────────────────────────────────
print("\n  [4b] Reconocimiento de entidades nombradas (NER)...")

# Modelo multilingüe con buen soporte para español
# (mismo paper: Babelscape/wikineural-multilingual-ner vía Transformers)
from transformers import pipeline

ner_pipeline = pipeline(
    "ner",
    model="Babelscape/wikineural-multilingual-ner",
    aggregation_strategy="simple",
    device=0 if DEVICE == "cuda" else -1,
)

def extraer_entidades(textos: list[str], batch_size: int = 64) -> list[list[dict]]:
    """Extrae entidades en lote; devuelve lista de listas de entidades."""
    resultados = []
    for i in range(0, len(textos), batch_size):
        lote = textos[i : i + batch_size]
        # Truncar a 512 tokens para el modelo
        lote_trunc = [t[:512] for t in lote]
        out = ner_pipeline(lote_trunc)
        resultados.extend(out)
        if (i // batch_size) % 20 == 0:
            print(f"      NER: {i + len(lote):,}/{len(textos):,} tweets procesados")
    return resultados

textos_ner = df["texto_limpio"].tolist()
entidades_corpus = extraer_entidades(textos_ner)

# Construimos DataFrame de entidades
rows_ner = []
for tweet_id, ents in zip(df["tweet_id"].tolist(), entidades_corpus):
    tiene_salud_flag = 1 if tweet_id in ids_salud else 0
    for ent in ents:
        rows_ner.append({
            "tweet_id"   : tweet_id,
            "entidad"    : ent["word"].strip(),
            "tipo"       : ent["entity_group"],
            "score"      : round(ent["score"], 4),
            "tiene_salud": tiene_salud_flag,
        })

df_ner = pd.DataFrame(rows_ner)
print(f"    Entidades extraídas: {len(df_ner):,}")
print(f"    Tipos encontrados  : {df_ner['tipo'].unique()}")

def tabla_frecuencia_entidades(df_ner: pd.DataFrame, tipo: str, top_n: int = 15) -> pd.DataFrame:
    """
    Replicamos la Tabla 5/6/7 del paper:
    frecuencia en corpus total vs. subcorpus de salud + ratio.
    """
    sub = df_ner[df_ner["tipo"] == tipo]
    frec_corpus = sub.groupby("entidad")["tweet_id"].nunique().rename("freq_corpus")
    frec_salud  = (sub[sub["tiene_salud"] == 1]
                   .groupby("entidad")["tweet_id"].nunique()
                   .rename("freq_salud"))
    tabla = pd.concat([frec_corpus, frec_salud], axis=1).fillna(0).astype(int)
    tabla["pct_salud_corpus"] = (tabla["freq_salud"] / tabla["freq_corpus"] * 100).round(1)
    tabla = tabla.sort_values("freq_corpus", ascending=False).head(top_n)
    return tabla

for tipo_ent in ["PER", "ORG", "LOC"]:
    tabla = tabla_frecuencia_entidades(df_ner, tipo_ent)
    nombre_archivo = OUTPUT_DIR / f"ner_top_{tipo_ent.lower()}.csv"
    tabla.to_csv(nombre_archivo)
    print(f"\n    Top entidades ({tipo_ent}):")
    print(tabla.to_string())

# ─────────────────────────────────────────────────────────────────────────────
# 4c. MODELADO DE TÓPICOS (BERTopic)
# ─────────────────────────────────────────────────────────────────────────────
print("\n  [4c] Modelado de tópicos con BERTopic...")

from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

# Vectorizador con stopwords en español
vectorizer = CountVectorizer(
    stop_words=list(STOPWORDS),
    min_df=5,
    ngram_range=(1, 2),
)

# Configuración UMAP y HDBSCAN (igual que BERTopic internamente)
umap_model  = UMAP(n_neighbors=15, n_components=5, min_dist=0.0,
                   metric="cosine", random_state=SEED)
hdbscan_model = HDBSCAN(min_cluster_size=15, metric="euclidean",
                         cluster_selection_method="eom", prediction_data=True)

def entrenar_bertopic(textos: list[str], embeddings: np.ndarray,
                       num_topics: str | int, etiqueta: str) -> tuple:
    """Entrena BERTopic y devuelve (modelo, tópicos, probs, coherencia_cv)."""
    nr_topics = num_topics if num_topics == "auto" else int(num_topics)
    topic_model = BERTopic(
        embedding_model=modelo_emb,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer,
        nr_topics=nr_topics,
        calculate_probabilities=False,
        verbose=False,
    )
    topics, _ = topic_model.fit_transform(textos, embeddings)
    n_topics_final = len(set(topics)) - (1 if -1 in topics else 0)
    print(f"    [{etiqueta}] num_topics={num_topics} → tópicos finales: {n_topics_final}")
    return topic_model, topics, n_topics_final

# ── Corpus completo ──────────────────────────────────────────────────────────
print("\n    → Corpus completo:")
configs_corpus = ["auto", 60, 40, 30, 20, 10]
resultados_corpus = []

textos_corpus_chunks = df_chunks["texto"].tolist()
emb_corpus = embeddings_chunks  # ya calculados

for cfg in configs_corpus:
    modelo_bt, topics_bt, n_final = entrenar_bertopic(
        textos_corpus_chunks, emb_corpus, cfg, "CORPUS"
    )
    resultados_corpus.append({
        "num_topics": cfg, "num_topics_final": n_final,
        "modelo": modelo_bt, "topics": topics_bt,
    })

# Seleccionamos el modelo con más tópicos (igual que el paper: máx. coherencia ≈ máx. tópicos)
mejor_corpus_idx = max(
    range(len(resultados_corpus)),
    key=lambda i: resultados_corpus[i]["num_topics_final"],
)
mejor_modelo_corpus = resultados_corpus[mejor_corpus_idx]["modelo"]
mejor_topics_corpus = resultados_corpus[mejor_corpus_idx]["topics"]

print(f"\n    Mejor configuración corpus: {resultados_corpus[mejor_corpus_idx]['num_topics']}")
print(mejor_modelo_corpus.get_topic_info().head(10).to_string(index=False))
mejor_modelo_corpus.get_topic_info().to_csv(
    OUTPUT_DIR / "topicos_corpus_completo.csv", index=False
)

# ── Subcorpus de salud ───────────────────────────────────────────────────────
print("\n    → Subcorpus de salud:")
idx_salud_chunks = df_chunks[df_chunks["es_salud"] == 1].index.tolist()
textos_salud_chunks = df_subcorpus["texto"].tolist()
emb_salud = embeddings_chunks[idx_salud_chunks]

configs_salud = ["auto", 40, 30, 20, 10]
resultados_salud = []

for cfg in configs_salud:
    modelo_bt_s, topics_bt_s, n_final_s = entrenar_bertopic(
        textos_salud_chunks, emb_salud, cfg, "SALUD"
    )
    resultados_salud.append({
        "num_topics": cfg, "num_topics_final": n_final_s,
        "modelo": modelo_bt_s, "topics": topics_bt_s,
    })

mejor_salud_idx = max(
    range(len(resultados_salud)),
    key=lambda i: resultados_salud[i]["num_topics_final"],
)
mejor_modelo_salud = resultados_salud[mejor_salud_idx]["modelo"]

print(f"\n    Mejor configuración salud: {resultados_salud[mejor_salud_idx]['num_topics']}")
print(mejor_modelo_salud.get_topic_info().head(10).to_string(index=False))
mejor_modelo_salud.get_topic_info().to_csv(
    OUTPUT_DIR / "topicos_subcorpus_salud.csv", index=False
)

# ─────────────────────────────────────────────────────────────────────────────
# 4d. ANÁLISIS DE VOCABULARIO CON POS-TAGGING
# ─────────────────────────────────────────────────────────────────────────────
print("\n  [4d] Análisis de vocabulario con POS-Tagging (Stanza)...")

# Reutilizamos el pipeline de Stanza ya cargado (con POS)
TAGS_INTERES = {"VERB", "ADJ", "NOUN"}

def extraer_pos(textos: list[str], batch_size: int = 512) -> list[dict]:
    """
    Extrae verbos, adjetivos y sustantivos con su lema.
    Devuelve lista de dicts {lema, upos, tweet_idx}.
    """
    registros = []
    for i in range(0, len(textos), batch_size):
        lote  = textos[i : i + batch_size]
        docs  = [stanza.Document([], text=t) for t in lote]
        docs  = nlp_stanza(docs)
        for j, doc in enumerate(docs):
            for sent in doc.sentences:
                for word in sent.words:
                    if word.upos in TAGS_INTERES and word.lemma:
                        registros.append({
                            "tweet_idx": i + j,
                            "lema"     : word.lemma.lower(),
                            "pos"      : word.upos,
                        })
        if (i // batch_size) % 10 == 0:
            print(f"      POS: {i + len(lote):,}/{len(textos):,} tweets procesados")
    return registros

textos_pos = df["texto_limpio"].tolist()
registros_pos = extraer_pos(textos_pos)
df_pos = pd.DataFrame(registros_pos)

# Marcar si el tweet pertenece al subcorpus de salud
ids_salud_list = sorted(ids_salud)
tweet_idx_a_salud = {
    idx: (1 if df.iloc[idx]["tiene_salud"] == 1 else 0)
    for idx in range(len(df))
}
df_pos["tiene_salud"] = df_pos["tweet_idx"].map(tweet_idx_a_salud)

UMBRAL_PROP = len(df_subcorpus) / len(df_chunks)  # ≈ proporción de fragmentos de salud

def tabla_pos(df_pos: pd.DataFrame, pos_tag: str, top_n: int = 20) -> pd.DataFrame:
    """
    Replicamos las Tablas 12/13/14 del paper:
    frecuencia total, frecuencia en salud, proporción en salud.
    """
    sub = df_pos[df_pos["pos"] == pos_tag]
    frec_total = sub.groupby("lema").size().rename("freq_total")
    frec_salud = sub[sub["tiene_salud"] == 1].groupby("lema").size().rename("freq_salud")
    tabla = pd.concat([frec_total, frec_salud], axis=1).fillna(0).astype(int)
    tabla["prop_salud"] = (tabla["freq_salud"] / tabla["freq_total"]).round(3)
    # Filtramos palabras con proporción mayor a la base de salud
    tabla = tabla[tabla["prop_salud"] > UMBRAL_PROP]
    tabla = tabla.sort_values("freq_total", ascending=False).head(top_n)
    return tabla

for tag, nombre in [("VERB", "verbos"), ("ADJ", "adjetivos"), ("NOUN", "sustantivos")]:
    tabla = tabla_pos(df_pos, tag)
    ruta = OUTPUT_DIR / f"vocabulario_{nombre}.csv"
    tabla.to_csv(ruta)
    print(f"\n    Top {nombre} más asociados a salud:")
    print(tabla.head(10).to_string())


PASO 4 – ANÁLISIS COMPARATIVO

  [4a] Análisis descriptivo y exploratorio...

    ESTADÍSTICAS GENERALES DEL CORPUS
    ---------------------------------------------
    Total tweets                            : 151424
    Total palabras                          : 2440197
    Media palabras por tweet                : 16.1
    Mediana palabras por tweet              : 16.0
    Máximo palabras                         : 39
    Mínimo palabras                         : 1
    Autores distintos                       : 79938
    Tweets con mención de salud             : 22194
    % tweets con mención de salud           : 14.7
    Nube guardada: outputs\nube_corpus_completo.png
    Nube guardada: outputs\nube_subcorpus_salud.png
    Figura temporal guardada.

  [4b] Reconocimiento de entidades nombradas (NER)...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/709M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: Babelscape/wikineural-multilingual-ner
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\afpue\AppData\Roaming\Python\Python311\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\afpue\AppData\Roaming\Python\Python311\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\afpue\AppData\Roaming\Python\Python311\site-packages\ipykernel\kernelapp.py", line 758, in start
    self.io_lo

AttributeError: _ARRAY_API not found

ImportError: numpy.core._multiarray_umath failed to import

ImportError: numpy.core.umath failed to import

TypeError: Unable to convert function return value to a Python type! The signature was
	() -> handle

In [ ]:
# =============================================================================
# PASO 5 – INTERPRETACIÓN Y DISCUSIÓN
# =============================================================================
print("\n" + "=" * 70)
print("PASO 5 – INTERPRETACIÓN Y DISCUSIÓN")
print("=" * 70)

# Resumen ejecutivo automático para orientar la interpretación
print(f"""
  RESUMEN DE HALLAZGOS
  ─────────────────────────────────────────────────────────────
  · Corpus total            : {len(df):,} tweets
  · Con mención de salud    : {n_tweets_salud:,} ({pct_tweets:.1f}%)
  · Categoría más frecuente : {dist_cats.iloc[0]['Categoría']} ({dist_cats.iloc[0]['Porcentaje']:.1f}%)
  · Umbral τ utilizado      : {TAU:.2f}
  · Modelo embeddings       : {MODELO_EMBEDDINGS}
  · Tópicos corpus completo : {resultados_corpus[mejor_corpus_idx]['num_topics_final']}
  · Tópicos subcorpus salud : {resultados_salud[mejor_salud_idx]['num_topics_final']}
  ─────────────────────────────────────────────────────────────
  Archivos generados en: {OUTPUT_DIR.resolve()}
""")

print("  PROCESO COMPLETADO ✓")

[CONFIG] Dispositivo: cuda

PASO 1 – RECOLECCIÓN DE DATOS
  Tweets cargados  : 151,424
  Columnas         : ['Author', 'Content', 'Date', 'Location', 'Number of Likes', 'Number of Retweets', 'In Reply To', 'Author Name', 'Author Description', 'Author Statuses Count', 'Author Favourites Count', 'Author Friends Count', 'Author Followers Count', 'Author Listed Count', 'Author Verified', 'Mentions', 'Hashtags', 'Content_cleaned', 'Content_cleaned_2', 'Seed_Set_1', 'In Reply To Normalized', 'Seed_Set_2', 'Seed_Set_2_Prob', 'Seed_Set_2_Model', 'Author_Normalized', 'Seed_Set_3', 'Entidad', 'Prob entidad', 'Polaridad', 'Fecha', 'Sentimiento']
  Rango de fechas  : 2020-03-31 → 2020-08-18

PASO 2 – PREPROCESAMIENTO

  [2.1] Limpieza de texto (URLs, menciones, hashtags, caracteres especiales)...
    Tweets con texto vacío tras limpieza: 0
    Tweets conservados                  : 151,424

  [2.2] Tokenización...
    Tokens totales: 2,440,208

  [2.3] Eliminación de stopwords...
    Stopwords conf

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

    Modelo cargado: paraphrase-multilingual-mpnet-base-v2 | Dispositivo: cuda

  [3.3] Segmentación en fragmentos...
    Fragmentos generados: 151,424 (de 151,424 tweets)

  [3.4] Generación de embeddings para fragmentos y categorías del Tesauro...
    Codificando 151,424 fragmentos (batch=256)...


Batches:   0%|          | 0/592 [00:00<?, ?it/s]

    Shape embeddings fragmentos: (151424, 768)
    Shape embeddings categorías: (12, 768)

  [3.5] Cómputo de la matriz de similitud coseno S ∈ R^(m × n)...
    Matriz S: (151424, 12)  →  151,424 fragmentos × 12 categorías
    Similitud máxima global : 0.8283
    Similitud media global  : 0.1868

  [3.6] Determinación del umbral τ mediante validación manual...
    Distribución de sim_max:
      P 10: 0.1930
      P 25: 0.2417
      P 50: 0.3193
      P 75: 0.4197
      P 90: 0.4975
      P 95: 0.5405
      P 99: 0.6180


KeyError: "['cat_nombre'] not in index"

In [ ]:
import torch

print("PyTorch version :", torch.__version__)
print("CUDA disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU             :", torch.cuda.get_device_name(0))
    print("Memoria total   :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
    # Prueba rápida de velocidad
    import time
    a = torch.randn(10000, 10000, device="cuda")
    b = torch.randn(10000, 10000, device="cuda")
    t0 = time.time()
    c = a @ b
    torch.cuda.synchronize()
    print(f"Multiplicación 10k×10k: {(time.time()-t0)*1000:.1f} ms en GPU")
else:
    print("⚠ No se detectó GPU. Corriendo en CPU.")

PyTorch version : 2.6.0+cu124
CUDA disponible : True
GPU             : NVIDIA GeForce RTX 4050 Laptop GPU
Memoria total   : 6.4 GB
Multiplicación 10k×10k: 604.2 ms en GPU
